# 02 — Preprocessamento e Label Engineering


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
tqdm.pandas()

from src.data.loader import load_raw, save_processed
from src.data.preprocessor import derive_labels, add_category_labels

## 1. Carregar dados brutos

In [ ]:
# Ajuste o nome do arquivo se necessário
RAW_PATH = Path('../data/raw')
csv_files = list(RAW_PATH.glob('*.csv'))
df = pd.read_csv(csv_files[0], low_memory=False)
print(f'Shape original: {df.shape}')

## 2. Derivar labels e texto limpo

In [ ]:
# Remover linhas sem score
if 'Reviewer_Score' in df.columns:
    df = df.dropna(subset=['Reviewer_Score'])
    df['Reviewer_Score'] = pd.to_numeric(df['Reviewer_Score'], errors='coerce')
    df = df.dropna(subset=['Reviewer_Score'])

print(f'Após limpeza inicial: {df.shape}')

# Derivar labels
print('Derivando labels...')
df_labeled = derive_labels(df)
print(f'Colunas após derive_labels: {df_labeled.columns.tolist()}')

In [ ]:
# Remover reviews com texto vazio após limpeza
df_labeled = df_labeled[df_labeled['text'].str.len() > 20].reset_index(drop=True)
print(f'Após filtro de texto vazio: {df_labeled.shape}')

# Distribuição de sentimento
sent_dist = df_labeled['sentiment_label'].value_counts()
sent_names = {0: 'negativo', 1: 'neutro', 2: 'positivo'}
print('\nDistribuição de sentimento:')
for k, v in sent_dist.items():
    print(f'  {sent_names[k]}: {v:,} ({v/len(df_labeled)*100:.1f}%)')

## 3. Classificar categorias (keyword matching)

In [ ]:
print('Classificando categorias...')
df_labeled = add_category_labels(df_labeled)

cat_cols = [c for c in df_labeled.columns if c.startswith('cat_')]
cat_sums = df_labeled[cat_cols].sum().sort_values(ascending=False)
print('\nMenções por categoria:')
print(cat_sums)

## 4. Verificar qualidade e balanceamento

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sentimento
sent_counts = df_labeled['sentiment_label'].map(sent_names).value_counts()
axes[0].bar(sent_counts.index, sent_counts.values)
axes[0].set_title('Distribuição Sentimento')

# Prioridade
prio_counts = df_labeled['priority_label'].value_counts()
axes[1].bar(['normal', 'alta'], [prio_counts.get(0,0), prio_counts.get(1,0)])
axes[1].set_title('Distribuição Prioridade')

# Idioma
if 'language' in df_labeled.columns:
    lang_counts = df_labeled['language'].value_counts()
    axes[2].bar(lang_counts.index, lang_counts.values)
    axes[2].set_title('Idiomas')

plt.tight_layout()
plt.show()

## 5. Salvar dados processados

In [ ]:
save_processed(df_labeled, 'reviews_labeled.csv')
print(f'Salvo em data/processed/reviews_labeled.csv')
print(f'Shape final: {df_labeled.shape}')
df_labeled.head(3)